In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox

In [25]:
# Load data
df = pd.read_csv("../data/raw/DCOILWTICO.csv")

# Convert date
df["observation_date"] = pd.to_datetime(df["observation_date"])

# Remove missing prices first
price = (
    df.loc[
        df["DCOILWTICO"].notna(),
        ["observation_date", "DCOILWTICO"]
    ]
    .set_index("observation_date")["DCOILWTICO"]
)

# Calculate simple returns
return_series = price.pct_change().dropna()

# Scale returns to percentage
garch_data = return_series * 100

print(garch_data.describe())
print("Missing values:", garch_data.isna().sum())
print("Infinite values:", np.isinf(garch_data).sum())

count    10225.000000
mean         0.013765
std          4.217723
min       -301.966139
25%         -1.210018
50%          0.074710
75%          1.327434
max         53.086420
Name: DCOILWTICO, dtype: float64
Missing values: 0
Infinite values: 0


In [26]:
split_date = "2023-01-01"

train = garch_data[garch_data.index < split_date]
test = garch_data[garch_data.index >= split_date]

print("Training period:")
print(train.index.min(), "to", train.index.max())

print("\nTest period:")
print(test.index.min(), "to", test.index.max())

print("\nTraining observations:", len(train))
print("Test observations:", len(test))

Training period:
1986-01-03 00:00:00 to 2022-12-30 00:00:00

Test period:
2023-01-03 00:00:00 to 2026-08-18 00:00:00

Training observations: 9322
Test observations: 903


In [27]:
model_specs = {
    "GARCH(1,1)": {"p": 1, "q": 1},
    "GARCH(1,2)": {"p": 1, "q": 2},
    "GARCH(2,1)": {"p": 2, "q": 1},
    "GARCH(2,2)": {"p": 2, "q": 2}
}

In [28]:
results = {}

for name, spec in model_specs.items():

    model = arch_model(
        train,
        mean="Constant",
        vol="GARCH",
        p=spec["p"],
        q=spec["q"],
        dist="normal"
    )

    result = model.fit(disp="off")

    results[name] = result

    print(f"{name} fitted.")

GARCH(1,1) fitted.
GARCH(1,2) fitted.
GARCH(2,1) fitted.
GARCH(2,2) fitted.


In [31]:
comparison = pd.DataFrame({
    name: {
        "AIC": result.aic,
        "BIC": result.bic,
        "Log-Likelihood": result.loglikelihood
    }
    for name, result in results.items()
}).T

comparison = comparison.sort_values("AIC")

comparison["Delta_AIC"] = comparison["AIC"] - comparison["AIC"].min()

comparison

,AIC,BIC,Log-Likelihood,Delta_AIC
"GARCH(2,2)",41870.269609,41913.110404,-20929.134805,0.000000
"GARCH(1,1)",41894.640417,41923.200947,-20943.320209,24.370808
"GARCH(1,2)",41895.974175,41931.674838,-20942.987088,25.704566
"GARCH(2,1)",41896.640417,41932.341079,-20943.320208,26.370808


In [32]:
#Standardized Residual Diagnostics

standardized_residuals = {}

for name, result in results.items():
    standardized_residuals[name] = result.std_resid.dropna()


#Ljung-Box
residual_lb_results = {}

for name, resid in standardized_residuals.items():

    lb = acorr_ljungbox(
        resid,
        lags=[10, 20],
        return_df=True
    )

    residual_lb_results[name] = lb

In [33]:
residual_lb_table = pd.DataFrame({
    name: {
        "LB(10) p-value": lb.loc[10, "lb_pvalue"],
        "LB(20) p-value": lb.loc[20, "lb_pvalue"]
    }
    for name, lb in residual_lb_results.items()
}).T

residual_lb_table

,LB(10) p-value,LB(20) p-value
"GARCH(1,1)",0.716147,0.478893
"GARCH(1,2)",0.712235,0.471626
"GARCH(2,1)",0.716149,0.478895
"GARCH(2,2)",0.684840,0.431731


In [35]:
#GARCH 有沒有成功把 volatility clustering 解釋掉？
squared_residual_lb_results = {}

for name, resid in standardized_residuals.items():

    lb = acorr_ljungbox(
        resid ** 2,
        lags=[10, 20],
        return_df=True
    )

    squared_residual_lb_results[name] = lb

squared_residual_lb_table = pd.DataFrame({
    name: {
        "Squared LB(10) p-value": lb.loc[10, "lb_pvalue"],
        "Squared LB(20) p-value": lb.loc[20, "lb_pvalue"]
    }
    for name, lb in squared_residual_lb_results.items()
}).T

squared_residual_lb_table



,Squared LB(10) p-value,Squared LB(20) p-value
"GARCH(1,1)",0.999907,0.999989
"GARCH(1,2)",0.999922,0.999989
"GARCH(2,1)",0.999907,0.999989
"GARCH(2,2)",0.999697,0.999983


In [36]:
final_comparison = comparison.join(residual_lb_table)
final_comparison = final_comparison.join(squared_residual_lb_table)

final_comparison

,AIC,BIC,Log-Likelihood,Delta_AIC,LB(10) p-value,LB(20) p-value,Squared LB(10) p-value,Squared LB(20) p-value
"GARCH(2,2)",41870.269609,41913.110404,-20929.134805,0.000000,0.684840,0.431731,0.999697,0.999983
"GARCH(1,1)",41894.640417,41923.200947,-20943.320209,24.370808,0.716147,0.478893,0.999907,0.999989
"GARCH(1,2)",41895.974175,41931.674838,-20942.987088,25.704566,0.712235,0.471626,0.999922,0.999989
"GARCH(2,1)",41896.640417,41932.341079,-20943.320208,26.370808,0.716149,0.478895,0.999907,0.999989


In [37]:
#選擇GARCH(2,2)

final_model = arch_model(
    train,
    mean="Constant",
    vol="GARCH",
    p=2,
    q=2,
    dist="normal"
)

final_result = final_model.fit(disp="off")

print(final_result.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:             DCOILWTICO   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -20929.1
Distribution:                  Normal   AIC:                           41870.3
Method:            Maximum Likelihood   BIC:                           41913.1
                                        No. Observations:                 9322
Date:                Sat, Sep 05 2026   Df Residuals:                     9321
Time:                        20:31:34   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0583  2.327e-02      2.506  1.221e-02 [1.271e-0

In [38]:
final_result.params

mu          5.831597e-02
omega       1.119280e-01
alpha[1]    1.249168e-01
alpha[2]    6.385765e-02
beta[1]     1.116044e-07
beta[2]     8.112259e-01
Name: params, dtype: float64

In [39]:
alpha = final_result.params["alpha[1]"]
beta = final_result.params["beta[1]"]

persistence = alpha + beta

print("Alpha:", alpha)
print("Beta:", beta)
print("Alpha + Beta:", persistence)

Alpha: 0.12491682185731208
Beta: 1.1160443139314398e-07
Alpha + Beta: 0.12491693346174347
